# Initial Setup and Dataset Audit

This notebook performs the initial dataset audit and creates the shared test splits. It is designed to run in Google Colab.

## 1. Verify GPU Activation

Make sure you have enabled a GPU runtime (Runtime → Change runtime type → GPU).

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Devices:", tf.config.list_physical_devices("GPU"))

## 2. Mount Google Drive

Mount your Google Drive to access the dataset and save the generated manifests and reports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

Set the paths to your dataset and repository in Google Drive.

In [ ]:
# --- CHANGE THESE PATHS TO MATCH YOUR DRIVE STRUCTURE ---

GITHUB_REPO_URL = "https://github.com/MyElhadri/histology-ai-classification.git"

# Path where you want to clone the repo in Colab
REPO_DIR = "/content/histology-ai-classification"

# Path to the raw dataset in your Google Drive
DATASET_ROOT = "/content/drive/MyDrive/histology_project/dataset/Human_Histopathological_H_E_Stained_Nuclei_Images"

# Path where outputs (manifests, reports) should be saved in your Google Drive
OUTPUT_ROOT = "/content/drive/MyDrive/histology_project/outputs"

## 4. Clone Repository & Install Dependencies

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository into {REPO_DIR}...")
    subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("Installing requirements...")
subprocess.run(["pip", "install", "-r", f"{REPO_DIR}/requirements-colab.txt"], check=True)
print("Setup complete.")

## 5. Run Dataset Audit and Create Splits

In [ ]:
import sys
# Add repo to python path so we can import local modules if needed
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Run the prepare_data script
!python {REPO_DIR}/scripts/prepare_data.py --dataset-root "{DATASET_ROOT}" --output-root "{OUTPUT_ROOT}"

## 6. View Results

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

print("=== Audit Report ===")
try:
    with open(f"{OUTPUT_ROOT}/reports/metrics/dataset_audit.json", "r") as f:
        audit_report = json.load(f)
        print(json.dumps(audit_report, indent=2))
except FileNotFoundError:
    print("Audit report not found.")

print("\n=== Class Distribution ===")
try:
    display(Image(filename=f"{OUTPUT_ROOT}/reports/figures/class_distribution.png"))
except FileNotFoundError:
    print("Class distribution plot not found.")

print("\n=== First 5 lines of Test Manifest ===")
try:
    test_df = pd.read_csv(f"{OUTPUT_ROOT}/data/manifests/test_manifest.csv")
    display(test_df.head())
except FileNotFoundError:
    print("Test manifest not found.")

print("\n=== First 5 lines of DenseNet121 Folds Manifest ===")
try:
    folds_df = pd.read_csv(f"{OUTPUT_ROOT}/data/manifests/densenet121_folds.csv")
    display(folds_df.head())
except FileNotFoundError:
    print("Folds manifest not found.")